# Prototipo CNN — TAC cerebral (Normal vs Stroke)

 Clasificador de imágenes con red convolucional.


In [18]:
from pathlib import Path


DATA_DIR = Path("../data/raw/Brain_Data_Organised")


clases = [p.name for p in DATA_DIR.iterdir() if p.is_dir()]


conteo = {c: sum(1 for _ in (DATA_DIR / c).glob("*")) for c in clases}

print("Ruta existe:", DATA_DIR.exists())
print("Clases:", clases)
print("Conteo:", conteo)

Ruta existe: True
Clases: ['Normal', 'Stroke']
Conteo: {'Normal': 1551, 'Stroke': 950}


In [2]:
from PIL import Image
import numpy as np


ej_normal = next((DATA_DIR / "Normal").glob("*"))
ej_stroke = next((DATA_DIR / "Stroke").glob("*"))

for etiqueta, ruta in [("Normal", ej_normal), ("Stroke", ej_stroke)]:
    img = Image.open(ruta)
    arr = np.array(img)
    print(f"{etiqueta}: {ruta.name}")
    print(f"   modo PIL: {img.mode}  |  tamaño (ancho, alto): {img.size}  |  shape numpy: {arr.shape}")

Normal: 100 (1).jpg
   modo PIL: L  |  tamaño (ancho, alto): (650, 650)  |  shape numpy: (650, 650)
Stroke: 58 (1).jpg
   modo PIL: L  |  tamaño (ancho, alto): (650, 650)  |  shape numpy: (650, 650)


In [3]:
from collections import Counter

formas = Counter()         
corruptas = []              

for clase in clases:
    for ruta in (DATA_DIR / clase).glob("*"):
        try:
            with Image.open(ruta) as img:
                formas[(img.mode, img.size)] += 1
        except Exception as e:
            corruptas.append((ruta.name, str(e)))

print("Combinaciones (modo, tamaño) encontradas:")
for combo, n in formas.most_common():
    print(f"   {combo}: {n}")

print(f"\nImágenes ilegibles: {len(corruptas)}")

Combinaciones (modo, tamaño) encontradas:
   ('L', (650, 650)): 2501

Imágenes ilegibles: 0


In [5]:
import torch, torchvision
print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

2.14.0+cpu 0.29.0+cpu False


## Preprocesado

Pipeline de transforms que adapta cada TAC (`L`, 650×650, uint8) al contrato de entrada de ResNet-50: 224×224, 3 canales, `float32` normalizado a las estadísticas de ImageNet. Redimensionado sin recorte para preservar el campo completo (una lesión puede ubicarse en el borde); gris replicado a 3 canales idénticos (`r == g == b`) por exigencia de la primera capa convolucional, no por aporte cromático. Se emplea la API `v2` de torchvision: la conversión a tensor se desdobla en `ToImage` + `ToDtype(scale=True)` —`ToTensor` está deprecado desde 0.16 y su reescalado implícito es fuente de errores silenciosos—, dejando el escalado 0–255 → 0–1 explícito. Normalización con media/desvío de ImageNet (heredados del preentrenamiento, no estimados sobre el dataset propio). Los cinco pasos se consolidan en un único `Compose`, receta reutilizable en entrenamiento y en inferencia.

In [11]:
import torch
from torchvision.transforms import v2

resize = v2.Resize((224, 224))
gray3 = v2.Grayscale(num_output_channels=3)
to_image = v2.ToImage()
to_float = v2.ToDtype(torch.float32, scale=True)

In [12]:
img = Image.open("../data/raw/Brain_Data_Organised/Normal/100 (1).jpg")
t = to_float(to_image(gray3(resize(img))))
print(t.shape, t.dtype, t.min().item(), t.max().item())

torch.Size([3, 224, 224]) torch.float32 0.0 1.0


In [13]:
normalize = v2.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

In [14]:
img = Image.open("../data/raw/Brain_Data_Organised/Normal/100 (1).jpg")
t = normalize(to_float(to_image(gray3(resize(img)))))
print(t.shape, t.dtype)
print(round(t.min().item(), 3), round(t.max().item(), 3))

torch.Size([3, 224, 224]) torch.float32
-2.118 2.64


In [15]:
transform = v2.Compose([
    resize,
    gray3,
    to_image,
    to_float,
    normalize,
])

In [16]:
t = transform(img)
print(t.shape, t.dtype)
print(round(t.min().item(), 3), round(t.max().item(), 3))

torch.Size([3, 224, 224]) torch.float32
-2.118 2.64


In [19]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(root=DATA_DIR, transform=transform)

print(len(dataset))
print(dataset.classes)
print(dataset.class_to_idx)

2501
['Normal', 'Stroke']
{'Normal': 0, 'Stroke': 1}


In [21]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

train_idx, test_idx = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.targets,
    random_state=42,
)

train_ds = Subset(dataset, train_idx)
test_ds = Subset(dataset, test_idx)

print(len(train_ds), len(test_ds))

2000 501


In [22]:
from collections import Counter

print("train:", Counter(dataset.targets[i] for i in train_idx))
print("test: ", Counter(dataset.targets[i] for i in test_idx))

train: Counter({0: 1240, 1: 760})
test:  Counter({0: 311, 1: 190})


In [23]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(len(train_loader), len(test_loader))

63 16


In [24]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:8])

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([0, 0, 1, 1, 1, 0, 0, 0])


## Modelado

Clasificación por transfer learning con ResNet-50 preentrenada en ImageNet. Se congela el backbone convolucional —las features de bajo nivel (bordes, texturas, contornos) aprendidas sobre millones de imágenes son transferibles al dominio médico— y se reemplaza la cabeza original de 1000 clases por una capa lineal de 2 salidas (Normal/Stroke). Entrenar desde cero se descarta por volumen (2.501 imágenes no alimentan los millones de parámetros de una CNN profunda) y por hardware (CPU-only); con el backbone congelado, el cómputo se reduce a la cabeza. Evaluación centrada en `recall` sobre la clase positiva (Stroke), coherente con el criterio clínico del proyecto —un falso negativo (ictus no detectado) es el error costoso—.

In [25]:
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet50_Weights

model = models.resnet50(weights=ResNet50_Weights.DEFAULT)   # 1. red + pesos ImageNet

for param in model.parameters():                            # 2. congelar todo
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)               # 3. cabeza nueva: 1000 → 2

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\arii_/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:10<00:00, 9.95MB/s]


In [26]:
print([n for n, p in model.named_parameters() if p.requires_grad])

['fc.weight', 'fc.bias']


In [27]:
import torch.optim as optim

device = torch.device("cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [28]:
from tqdm import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} — loss: {running_loss / len(train_loader):.4f}")

Epoch 1/3: 100%|██████████| 63/63 [10:33<00:00, 10.06s/it]


Epoch 1 — loss: 0.6368


Epoch 2/3: 100%|██████████| 63/63 [11:23<00:00, 10.85s/it]


Epoch 2 — loss: 0.5727


Epoch 3/3: 100%|██████████| 63/63 [11:38<00:00, 11.09s/it]

Epoch 3 — loss: 0.5315


In [29]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Eval"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Normal", "Stroke"]))

Eval: 100%|██████████| 16/16 [02:17<00:00,  8.57s/it]


[[290  21]
 [103  87]]
              precision    recall  f1-score   support

      Normal       0.74      0.93      0.82       311
      Stroke       0.81      0.46      0.58       190

    accuracy                           0.75       501
   macro avg       0.77      0.70      0.70       501
weighted avg       0.76      0.75      0.73       501



In [30]:
weights = torch.tensor([0.806, 1.316]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [31]:
EXTRA_EPOCHS = 7

for epoch in range(EXTRA_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Weighted {epoch+1}/{EXTRA_EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Weighted epoch {epoch+1} — loss: {running_loss / len(train_loader):.4f}")

Weighted 1/7: 100%|██████████| 63/63 [09:55<00:00,  9.45s/it]


Weighted epoch 1 — loss: 0.5206


Weighted 2/7: 100%|██████████| 63/63 [09:18<00:00,  8.86s/it]


Weighted epoch 2 — loss: 0.4947


Weighted 3/7: 100%|██████████| 63/63 [10:50<00:00, 10.33s/it]


...

Weighted epoch 5 — loss: 0.4518


Weighted 6/7: 100%|██████████| 63/63 [10:42<00:00, 10.20s/it]


Weighted epoch 6 — loss: 0.4195


Weighted 7/7: 100%|██████████| 63/63 [10:43<00:00, 10.21s/it]

Weighted epoch 7 — loss: 0.4047


In [32]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Eval"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Normal", "Stroke"]))

Eval: 100%|██████████| 16/16 [01:58<00:00,  7.38s/it]

[[243  68]
 [ 34 156]]
              precision    recall  f1-score   support

      Normal       0.88      0.78      0.83       311
      Stroke       0.70      0.82      0.75       190

    accuracy                           0.80       501
   macro avg       0.79      0.80      0.79       501
weighted avg       0.81      0.80      0.80       501



In [33]:
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

torch.save(model.state_dict(), MODELS_DIR / "cnn_resnet50_stroke.pth")

print("guardado:", (MODELS_DIR / "cnn_resnet50_stroke.pth").exists())

guardado: True
